# GeoRS-Agent — evaluation on Google Colab

**Before starting:** *Runtime → Change runtime type → T4 GPU*.

Put the image archive in a Google Drive folder (default `MyDrive/GeoRS-Agent/`):
- `WHU-RS19_ABZSL.zip`: the WHU-RS19 images, either in class folders (`Airport/`, `Beach/`, ...) or in `Train/` + `Test/`

The code is cloned from the tagged GitHub release (or taken from `georsagent.zip` in the same folder, if present).

All results are written **to Drive**, so if the session drops: reconnect,
rerun cells 1–5, then go back to the step that was interrupted.
It resumes where it stopped, because records already logged are skipped.

In [ ]:
#@title 1 · GPU and Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/GeoRS-Agent'   #@param {type:"string"}
MODEL       = 'qwen2.5vl:7b'                          #@param {type:"string"}
RESULTS_DIR = f'{DRIVE_DIR}/results_{MODEL.replace(":", "_").replace("/", "_")}'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print('results ->', RESULTS_DIR)

In [ ]:
#@title 2 · Unpack code and images
import os, glob
DATASET_ZIP = f'{DRIVE_DIR}/WHU-RS19_ABZSL.zip'   #@param {type:"string"}
assert os.path.exists(DATASET_ZIP), f'dataset zip not found: {DATASET_ZIP}'
REPO_URL = 'https://github.com/gapinnovation/GeoRS-Agent'   #@param {type:"string"}
REPO_TAG = 'v1.0.0'                                           #@param {type:"string"}
!rm -rf /content/georsagent
if os.path.exists(f'{DRIVE_DIR}/georsagent.zip'):     # option A: code zip on Drive
    !unzip -q -o "{DRIVE_DIR}/georsagent.zip" -d /content
else:                                                  # option B: tagged GitHub release
    !git clone -q --branch {REPO_TAG} {REPO_URL} /content/georsagent
!rm -rf /content/images && unzip -q -o "{DATASET_ZIP}" -d /content/images
# root = folder containing the class folders (Airport/, Beach/, ...) or Train/ + Test/
cands = [d for d in glob.glob('/content/images/**/', recursive=True)
         if os.path.isdir(os.path.join(d, 'Airport')) or
            (os.path.isdir(os.path.join(d, 'Train')) and os.path.isdir(os.path.join(d, 'Test')))]
if not cands:
    !find /content/images -maxdepth 3 -type d | head -30
assert cands, 'class folders not found: look at the folder list above'
IMAGES_ROOT = sorted(cands, key=len)[0]
print('images_root =', IMAGES_ROOT, '| contents:', sorted(os.listdir(IMAGES_ROOT))[:20])
%cd /content/georsagent
# Colab already provides pandas/numpy/scipy/requests: install only what is missing,
# without upgrading Colab's packages. Exact versions are logged in each run's manifest.
!pip -q install jsonschema tabulate

In [ ]:
#@title 3 · Install and start Ollama
!sudo apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
for _ in range(60):
    try:
        print('Ollama', requests.get('http://localhost:11434/api/version').json()); break
    except Exception: time.sleep(1)

In [ ]:
#@title 4 · Download the model (~5–6 GB, a few minutes)
!ollama pull {MODEL}
!ollama list

In [ ]:
#@title 5 · Configuration (results on Drive) + data preparation
import yaml, shutil
cfg = yaml.safe_load(open('config.yaml'))
cfg['paths']['images_root'] = IMAGES_ROOT
cfg['paths']['results_dir'] = RESULTS_DIR
cfg['ollama']['model'] = MODEL
cfg['interventions']['n_images'] = 95     # 5 per class for the model part (KB-level part already done on 301)
frozen = f'{RESULTS_DIR}/config_frozen.yaml'
if os.path.exists(frozen):               # resume: always reuse the SAME frozen config
    cfg = yaml.safe_load(open(frozen)); cfg['paths']['images_root'] = IMAGES_ROOT
    print('resume: reusing', frozen)
else:
    yaml.safe_dump(cfg, open(frozen, 'w'), sort_keys=False)
yaml.safe_dump(cfg, open('config_colab.yaml', 'w'), sort_keys=False)
!python scripts/prepare.py --config config_colab.yaml

import pandas as pd
t = pd.read_csv('data/processed/annotations.csv')
import sys; sys.path.insert(0, '.')
from georsagent.pipeline import resolve_image
missing = []
for f in t.filepath:
    try: resolve_image(IMAGES_ROOT, f)
    except FileNotFoundError: missing.append(f)
print('missing images:', len(missing), missing[:5])
assert not missing, 'some files listed in the workbook are not in the zip: check case/folder names'

## Smoke test and time estimate
One image per class, primary run only. Read the time estimate before starting the full run.

In [ ]:
#@title 6 · Smoke test (19 images × B0, B1, B2)
!python scripts/run_eval.py --config config_colab.yaml --runs primary --conditions B0 B1 B2 --limit 1

import json, numpy as np
recs = [json.loads(l) for l in open(f'{RESULTS_DIR}/eval_runs.jsonl')]
def secs(r):
    s = sum(a['meta'].get('wall_seconds', 0) for a in r.get('attempts', []) + r.get('attr_attempts', []))
    ex = r.get('explanation') or {}
    return s + sum(a['meta'].get('wall_seconds', 0) for a in ex.get('attempts', []))
per = {c: np.mean([secs(r) for r in recs if r['condition'] == c]) for c in ['B0', 'B1', 'B2']}
print({k: round(v, 1) for k, v in per.items()}, 's per image')
tot_img = sum(per.values())
k = cfg['protocol']['stability_images_per_class']
print(f"primary (301 img): ~{301*tot_img/3600:.1f} h")
print(f"stability (3 run × {19*k} img): ~{3*19*k*tot_img/3600:.1f} h")
print("full JSON smoke-test records in", RESULTS_DIR)

In [ ]:
#@title 7 · Inspect a raw response (sanity check)
r = next(r for r in recs if r['condition'] == 'B2')
print('true:', r['true_label'], '| pred:', r['pred_label'])
print(r['attr_attempts'][0]['raw'][:800])
print(r.get('explanation', {}).get('explanation'))

## Full run
Each cell can be stopped and restarted: it resumes where it left off.
**Do not change `config_frozen.yaml` once you have started.**

In [ ]:
#@title 8 · Primary run: all 301 test images, B0/B1/B2 + Oracle
!python scripts/run_eval.py --config config_colab.yaml --runs primary

In [ ]:
#@title 9 · Stability runs (stratified subset, T=0.7, 3 seeds)
!python scripts/run_eval.py --config config_colab.yaml --runs stab1 stab2 stab3 --conditions B0 B1 B2

In [ ]:
#@title 10 · Interventions (explanations + irrelevant metadata on B1)
!python scripts/run_interventions.py --config config_colab.yaml

In [ ]:
#@title 11 · Metrics and summary
!python scripts/compute_metrics.py --config config_colab.yaml > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_DIR}/summary.md').read()))
print(open(f'{RESULTS_DIR}/interventions_summary.json').read())

In [ ]:
#@title 12 · Pack results for the paper/repo
!cd "{RESULTS_DIR}/.." && zip -qr "{os.path.basename(RESULTS_DIR)}.zip" "{os.path.basename(RESULTS_DIR)}"
print('Archive in', DRIVE_DIR)

## Additional experiments (exploratory, added after the primary analysis)
- **Experiment B**: B1 without constrained decoding (~25 min on T4)
- **Experiment A**: knowledge base built from the VLM's own attributes on the 704 train images (~3 h on T4); then re-scoring of the B2 test evidence already logged

Run cells 1–5 first. Both cells are resumable.

In [ ]:
#@title 13 · Experiment B: B1 without JSON schema
!nvidia-smi > "{RESULTS_DIR}/gpu_expB.txt"
!python scripts/run_eval.py --config config_colab.yaml --runs primary --conditions B1_NOSCHEMA

In [ ]:
#@title 14 · Experiment A: train-set attribute extraction + VLM knowledge base
!nvidia-smi > "{RESULTS_DIR}/gpu_expA.txt"
!python scripts/run_experiment_a.py --config config_colab.yaml
!python scripts/compute_metrics.py --config config_colab.yaml > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_DIR}/summary.md').read()))

## Interactive agent (tool-calling) evaluation
380 verifiable questions (4 types × 95 images), AGENT vs NOTOOLS baseline. ~2–2.5 h on T4. Requires the primary run (B2) to be complete. Resumable.

In [ ]:
#@title 15 · Interactive agent evaluation
!nvidia-smi > "{RESULTS_DIR}/gpu_agent.txt"
!python scripts/run_agent_eval.py --config config_colab.yaml
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_DIR}/agent_summary.md').read()))